# 01 - Ingest

Cloud-agnostic ingestion: pull data from S3, Azure Data Lake Storage Gen2, Google Cloud Storage, a plain public HTTPS bucket, or local disk through one common `IngestionConnector` interface (`utils/ingestion/`). Everything downstream only ever touches local files under `datasets/raw/` - it never needs to know which cloud a dataset came from.

This notebook: (1) demonstrates each connector's construction, (2) pulls the real `sen1floods11` data from this project's own S3 mirror (`s3://sphoorthq-geoverse`, `us-east-1`) into `datasets/raw/sen1floods11/` following the same folder layout as the original public source, falling back to the public GCS bucket if the S3 mirror isn't reachable from wherever this runs, (3) catalogs everything currently in `datasets/raw/`.

In [1]:
import os
import sys

# Portable project-root resolution (no machine-specific hardcoded path) -
# walk up from the current working directory until pyproject.toml is found,
# the same marker file utils/core/paths.py's Path(__file__)-based resolution
# implicitly relies on being importable from.
sys.path.insert(0, next(
    d for d in (
        os.path.abspath(os.path.join(os.getcwd(), *([os.pardir] * i)))
        for i in range(8)
    )
    if os.path.exists(os.path.join(d, "pyproject.toml"))
))

from utils.core.paths import RAW_DIR
from utils.ingestion.factory import get_connector
from utils.observability.run_logger import RunLogger

_root = sys.path[0]
logger = RunLogger("01_ingest")

# This project's own S3 mirror of sen1floods11 - source of truth. It's a
# straight, one-to-one copy of datasets/raw/sen1floods11/ (same relative
# paths), uploaded via S3Connector.upload_prefix() - so download_prefix()
# below needs only one prefix -> one local root, no per-subfolder key
# remapping. Every S3 path this platform touches (this raw data, plus the
# model registry + job/run history in utils/core/cloud_state.py) shares the
# same "datasets/..." layout under one bucket, s3://sphoorthq-geoverse.
S3_BUCKET = "sphoorthq-geoverse"
S3_REGION = "us-east-1"
S3_RAW_PREFIX = "datasets/raw/sen1floods11/"
SEN1FLOODS11_ROOT = RAW_DIR / "sen1floods11"
HAND_LABELED = SEN1FLOODS11_ROOT / "data" / "flood_events" / "HandLabeled"

# Original public source (no credentials needed) - used only as a fallback
# below if the S3 mirror isn't reachable from wherever this runs. Its layout
# differs from the S3 mirror's (top-level "v1.1/..." instead of
# "datasets/raw/sen1floods11/..."), so it needs its own per-prefix mapping
# into the same local folders utils/catalog/scanner.py and
# utils/ai/classic/sen1floods11_dataset.py expect.
GCS_FALLBACK_TARGETS = {
    "v1.1/splits/": SEN1FLOODS11_ROOT / "splits",
    "v1.1/data/flood_events/HandLabeled/LabelHand/": HAND_LABELED / "LabelHand",
    "v1.1/data/flood_events/HandLabeled/S1Hand/": HAND_LABELED / "S1Hand",
}


## Cloud connector examples

These are real, working connectors - not mocked. ADLS and GCS aren't executed here against live buckets because this environment has no credentials configured for them, but the code path is identical whether you run it here or in a credentialed environment: set the usual provider env vars and call `.list_objects()` / `.download_prefix()`. S3 *is* exercised for real below, against this project's own bucket.

In [2]:
# AWS S3 - this project's real bucket, credentials via AWS_ACCESS_KEY_ID/AWS_SECRET_ACCESS_KEY
# env vars, ~/.aws/credentials, or an IAM role (see the "Real pull" section below for the actual pull)
# s3 = get_connector("s3", bucket=S3_BUCKET, region_name=S3_REGION)

# Azure Data Lake Storage Gen2 - credentials via DefaultAzureCredential (env vars, managed identity, az login)
# adls = get_connector("adls", account_url="https://myaccount.dfs.core.windows.net", filesystem="sar-data")
# adls.download_prefix("sentinel1/", RAW_DIR / "sentinel1")

# Google Cloud Storage - credentials via Application Default Credentials, or anonymous=True for public buckets
# gcs = get_connector("gcs", bucket="my-sar-bucket")
# gcs.download_prefix("sentinel1/", RAW_DIR / "sentinel1")

print("ADLS/GCS examples above are real code, commented out because no cloud credentials are configured here.")


ADLS/GCS examples above are real code, commented out because no cloud credentials are configured here.


## Real pull: sen1floods11 from this project's S3 mirror

Primary source is `s3://sphoorthq-geoverse/datasets/raw/sen1floods11/` (`us-east-1`) via `S3Connector` -
standard boto3 credential resolution (env vars, `~/.aws/credentials`, or an IAM role), nothing hardcoded.
It's a one-to-one mirror of `datasets/raw/sen1floods11/`, so a single `download_prefix()` call restores
the exact folder structure `utils/catalog/scanner.py` and `utils/ai/classic/sen1floods11_dataset.py` expect -
no per-subfolder key remapping needed.

If this machine has no AWS credentials (or the bucket isn't reachable from here), this falls back to the
original public GCS bucket (`https://storage.googleapis.com/sen1floods11/`) - `v1.1/splits/`,
`v1.1/data/flood_events/HandLabeled/LabelHand/`, `v1.1/data/flood_events/HandLabeled/S1Hand/` - via the
no-credentials-needed HTTP connector, downloading for real (not just listing/verifying) so the fallback
path alone is enough to populate `datasets/raw/sen1floods11/` from nothing. Explicit and logged either
way, never a silent skip, so it's obvious from the stage metrics which source actually supplied the data.

In [3]:
def _download_from_s3() -> dict:
    s3 = get_connector("s3", bucket=S3_BUCKET, region_name=S3_REGION)
    downloaded = s3.download_prefix(S3_RAW_PREFIX, SEN1FLOODS11_ROOT, skip_existing=True)
    if not downloaded and not any(SEN1FLOODS11_ROOT.rglob("*.tif")):
        # An empty listing (bad prefix, empty bucket) looks identical to
        # "everything already present" - only treat it as real success if
        # local disk actually has data already, otherwise force the fallback.
        raise RuntimeError(f"s3://{S3_BUCKET}/{S3_RAW_PREFIX} returned no objects")
    return {"source": "s3", "bucket": S3_BUCKET, "prefix": S3_RAW_PREFIX, "files_downloaded": len(downloaded)}


def _download_from_gcs_http() -> dict:
    http = get_connector("http", base_url="https://storage.googleapis.com/sen1floods11/")
    downloaded = []
    for prefix, local_dir in GCS_FALLBACK_TARGETS.items():
        downloaded += http.download_prefix(prefix, local_dir, skip_existing=True)
    return {"source": "gcs_http_fallback", "files_downloaded": len(downloaded)}


with logger.stage("ingest_sen1floods11") as stage:
    try:
        stage.metrics = _download_from_s3()
        print(f"pulled from s3://{S3_BUCKET}/{S3_RAW_PREFIX}: {stage.metrics['files_downloaded']} files "
              f"(existing correctly-sized files skipped)")
    except Exception as e:
        print(f"S3 mirror not reachable from this machine ({type(e).__name__}: {e}) - "
              f"falling back to the public GCS bucket.")
        stage.metrics = _download_from_gcs_http()
        print(f"pulled from public GCS bucket: {stage.metrics['files_downloaded']} files "
              f"(existing correctly-sized files skipped)")


[01_ingest] -> ingest_sen1floods11 ...


pulled from s3://sphoorthq-geoverse/datasets/raw/sen1floods11/: 2235 files (existing correctly-sized files skipped)
[01_ingest] <- ingest_sen1floods11 [OK] 2.62s {'source': 's3', 'bucket': 'sphoorthq-geoverse', 'prefix': 'datasets/raw/sen1floods11/', 'files_downloaded': 2235}


## Catalog what's actually on local disk

`utils/catalog/scanner.py` walks `datasets/raw/` and reports real per-source metadata (dates, polarizations, chip counts, event names) - not placeholders.

In [4]:
from utils.catalog.scanner import scan_raw_datasets

with logger.stage("catalog_local_raw") as stage:
    records = scan_raw_datasets()
    stage.metrics = {"sources_found": len(records)}

for r in records:
    print(f"{r.id:45s} {r.source.value:14s} {r.status.value:10s}")

[01_ingest] -> catalog_local_raw ...
[01_ingest] <- catalog_local_raw [OK] 0.013s {'sources_found': 1}
sen1floods11                                  sen1floods11   raw       


In [5]:
logger.log_metrics({"local_sources": len(records)})
logger.finalize()

[01_ingest] run complete in 2.655s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\40ab7796-d485-4789-b547-9d521bdd9840.json


'D:\\project-raw-data\\sphoorthq-geoverse\\datasets\\reports\\runs\\40ab7796-d485-4789-b547-9d521bdd9840.json'